In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
#import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix


In [11]:
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=10, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=10, out_channels=10, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(490, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [12]:
train_transform = transforms.Compose([
    transforms.RandomRotation(degrees=10), 
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=test_transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [13]:
model = MNISTNet()
optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()


In [14]:
def train(model, train_loader, criterion, optimizer):
    model.train()
    t_loss = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        t_loss = t_loss + loss.item()

    return t_loss/len(train_loader)

In [15]:
def test(model, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(test_loader):
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

            # if batch_idx % 20 == 0:
            #   plt.imshow(data[0].view(28, 28).numpy(), cmap='gray')
            #   plt.title(f"Predicted: {pred[0].item()}, Actual: {target[0].item()}")
            #   plt.show()

    test_loss /= len(test_loader.dataset)

    print(f'Test accuracy: {100. * correct / len(test_loader.dataset):.4f}%')
    print(f'Test loss: {test_loss:.4f}')

In [16]:
num_epochs = 10
for epoch in range(1, num_epochs + 1):
    loss = train(model, train_loader, criterion, optimizer)

    print(f'Epoch: {epoch}, Loss: {loss:.4f}')

Epoch: 1, Loss: 0.2460
Epoch: 2, Loss: 0.1146
Epoch: 3, Loss: 0.0940
Epoch: 4, Loss: 0.0867
Epoch: 5, Loss: 0.0815
Epoch: 6, Loss: 0.0774
Epoch: 7, Loss: 0.0781
Epoch: 8, Loss: 0.0754
Epoch: 9, Loss: 0.0724
Epoch: 10, Loss: 0.0707


In [17]:
# device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# model.to(device)


test(model, test_loader)

# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# class_counts = torch.zeros(10)
# for _, labels in train_loader:
#     for label in labels:
#         class_counts[label] += 1

# print("Class distribution:", class_counts)

# # Add class-specific metrics
# all_preds = []
# all_labels = []

# with torch.no_grad():
#     for images, labels in test_loader:
#         images, labels = images.to(device), labels.to(device)
#         outputs = model(images)
#         _, preds = torch.max(outputs, 1)
#         all_preds.extend(preds.cpu().numpy())
#         all_labels.extend(labels.cpu().numpy())

# print("Classification Report for Digit 9:")
# print(classification_report(all_labels, all_preds, labels=[9]))
# print("Confusion Matrix:")
# print(confusion_matrix(all_labels, all_preds))
torch.save(model.state_dict(), "MNIST_CNN.pth")

Test accuracy: 98.6500%
Test loss: 0.0006
